In [ ]:
from typing import Any

import dataiku
import pandas as pd

In [ ]:
# Replace with the name of the target S3 connection to migrate all datasets to.
new_s3_connection = "dataiku-managed-storage"


In [ ]:
client = dataiku.api_client()
project_keys = client.list_project_keys()

In [ ]:
def get_all_dataset_metadata_for_project(project_handle: Any) -> pd.DataFrame:
    """Return a DataFrame of dataset connection metadata for a given project handle."""
    datasets = project_handle.list_datasets()
    if not datasets:
        return pd.DataFrame()
    extracted_data = [
        {
            'type': row.get('type'),
            'connection': row.get('params', {}).get('connection'),
            'name': row.get('name'),
            'table': row.get('params', {}).get('table'),
            'catalog': row.get('params', {}).get('catalog'),
            'schema': row.get('params', {}).get('schema'),
            'path': row.get('params', {}).get('path'),
        }
        for row in datasets
    ]
    return pd.DataFrame(extracted_data).sort_values(by=['type', 'connection', 'name'])


def check_connection_exists(connection_name: str) -> bool:
    """Return True if connection_name exists on this DSS instance."""
    return connection_name in client.list_connections()

In [ ]:
if check_connection_exists(new_s3_connection):
    for project_key in project_keys:
        project = client.get_project(project_key)
        df = get_all_dataset_metadata_for_project(project)

        if df.empty:
            print(f'Project {project_key} has no datasets')
            continue
        unmigrated_s3_connections = df[(df['type'] == 'S3') & (df['connection'] != new_s3_connection)]
        dataset_names_to_migrate = unmigrated_s3_connections['name'].unique().tolist()
        if not dataset_names_to_migrate:
            print(f"Project {project_key} has datasets, but none that need to be migrated")
        else:
            for dataset_name in dataset_names_to_migrate:
                dataset = project.get_dataset(dataset_name)
                settings = dataset.get_settings()
                settings.set_connection_and_path(new_s3_connection, settings.get_raw_params()['path'])
                settings.save()
                print(f"Project {project_key} dataset {dataset_name} updated to use connection: {new_s3_connection}")
else:
    print(f'Connection "{new_s3_connection}" does not exist on this instance — verify the connection name and try again.')

In [ ]:
for project_key in project_keys:
    project = client.get_project(project_key)
    df = get_all_dataset_metadata_for_project(project)
    print(df)